In [9]:

df = spark.read.table("mtomactual")
display(df)



StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6e1e4044-84de-4189-9fd8-bca7a0260fa0)

In [10]:
df.schema

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 14, Finished, Available, Finished, False)

StructType([StructField('Country', StringType(), True), StructField('Location', StringType(), True), StructField('Actual', StringType(), True)])

In [11]:
display(df.describe(["Country", "Actual"]).show())

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 15, Finished, Available, Finished, False)

+-------+-------+------------------+
|summary|Country|            Actual|
+-------+-------+------------------+
|  count|      6|                 6|
|   mean|   NULL| 7166.666666666667|
| stddev|   NULL|4020.7793606049395|
|    min|England|             11000|
|    max|  Italy|              7000|
+-------+-------+------------------+



In [12]:
df = df.select(df.Country, df.Location, df.Actual.cast("int"))
display(df)
display(df.describe(["Country", "Actual"]).show())

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f93b6462-5715-4d97-a053-3bfa8aa6fa73)

+-------+-------+------------------+
|summary|Country|            Actual|
+-------+-------+------------------+
|  count|      6|                 6|
|   mean|   NULL| 7166.666666666667|
| stddev|   NULL|4020.7793606049395|
|    min|England|              3000|
|    max|  Italy|             13000|
+-------+-------+------------------+



In [13]:
%%sql
SELECT Country, Location, CAST(Actual as INT)
FROM mtomactual

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 17, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 3 fields>

In [14]:
from pyspark.sql.types import *
schemaTarget = StructType([StructField('Country', StringType(), True), StructField('Location', StringType(), True), StructField('Actual', IntegerType(), True)])
df1 = spark.read.option("header","true").format("csv").schema(schemaTarget).load("Files/MtoMActual.csv")
display(df1)
df1.schema

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cfb19683-dcad-4f40-b85b-f0b0bf89425b)

StructType([StructField('Country', StringType(), True), StructField('Location', StringType(), True), StructField('Actual', IntegerType(), True)])

In [18]:
df1.write.format("delta").mode("overwrite").saveAsTable("mtomactualstruct")

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 23, Finished, Available, Finished, False)

In [19]:
from pyspark.sql.functions import *
df2 = spark.read.table("mtomactualwithdates")
df2 = df2.select(df2.Country, df2.Location, df2.Actual.cast("int"), \
date_format(df2.ColDate, "EEEE d MMMM yyyy").alias("FormateDate")
)
display(df2)

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4381655b-aa21-4233-a548-5b44142ea159)

In [20]:
%%sql
SELECT Country, Location, CAST(Actual as INT), date_format(colDate,"d MMMM yyyy") As FormattedDate
FROM mtomactualwithdates

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 25, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 4 fields>

In [25]:
df = spark.read.table("mtomactualstruct")
df.show()

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 32, Finished, Available, Finished, False)

+-------+----------+------+
|Country|  Location|Actual|
+-------+----------+------+
|England|    London|  5000|
|England|Birmingham|  7000|
|England|Manchester| 11000|
| France|     Paris|  4000|
|  Italy|     Milan|  3000|
|  Italy|      Rome| 13000|
+-------+----------+------+



In [26]:
display(df.groupBy("Country").sum("Actual").withColumnRenamed("sum(Actual)","ActualTotal"))

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb9b8524-3c8c-42e8-ad69-08830406e4c5)

In [28]:
display(df.groupBy("Country").sum("Actual").withColumnRenamed("sum(Actual)","ActualTotal")\
        .where("ActualTotal>10000")
)

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81dbf247-dbd9-43b5-b6b1-56879c386237)

In [29]:
%%sql
SELECT Country, SUM(Actual) AS ActualTotal
FROM mtomactualstruct
GROUP BY Country
HAVING SUM(Actual) > 10000

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 37, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [36]:
%%sql
SELECT *
FROM mtomactualstruct
-- ORDER BY Location ASC
ORDER By Country DESC , Location

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 53, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 3 fields>

In [30]:
df.show()

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 46, Finished, Available, Finished, False)

+-------+----------+------+
|Country|  Location|Actual|
+-------+----------+------+
|England|    London|  5000|
|England|Birmingham|  7000|
|England|Manchester| 11000|
| France|     Paris|  4000|
|  Italy|     Milan|  3000|
|  Italy|      Rome| 13000|
+-------+----------+------+



In [31]:
display(df.orderBy("Location"))

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2563b980-3191-40f5-af1f-d8fa93f522c2)

In [33]:
display(df.orderBy(desc("Location")))

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52cf6de2-702a-4aa0-a39a-382404508db8)

In [35]:
display(df.orderBy(desc(df.Country), df.Location))

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5a6d22e0-df88-45b9-8c99-42c700b8bd8a)

In [37]:
display(df.sort("Country"))

StatementMeta(, a5dc3cff-2d30-4cc6-b510-8ee1e2aa6db5, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8497e726-8e19-423e-809c-d881e755b002)